In [0]:
!pip install opencv-python

In [0]:
import os
import cv2
import glob


# 1) Define paths and parameters

# Manually specify the list of annotated video file paths (order matters!)
annotated_video_files = [
    # YOUR VIDEO PATH HERE
]

# Root output folder for decoded frames
output_root = "/xxx/Eem/Decoded Frames/videoxxxx_xxxx.mp4" # Replace with your output path
stride = 5                   # Extract 1 frame every 5 frames (adjust as needed)
max_frames_per_folder = 3000 # Maximum frames per subfolder, adjust as you need based on your Vram

# 2) Helper functions

def get_last_frame_index(output_root):
    """
    Scans existing subfolders under output_root and finds the highest frame index.
    Frames are named as 7-digit numbers (e.g., 0000001.jpg).
    """
    max_idx = 0
    if not os.path.exists(output_root):
        return max_idx
    # Loop over subfolders (e.g., video1, video2, etc.)
    for subfolder in os.listdir(output_root):
        subfolder_path = os.path.join(output_root, subfolder)
        if os.path.isdir(subfolder_path):
            for file in glob.glob(os.path.join(subfolder_path, "*.jpg")):
                basename = os.path.basename(file)
                name, ext = os.path.splitext(basename)
                try:
                    idx = int(name)
                    if idx > max_idx:
                        max_idx = idx
                except ValueError:
                    continue
    return max_idx

def get_output_folder(frame_index, output_root, max_frames_per_folder):
    """
    Determines the subfolder to use based on the current overall frame index.
    Each folder is named video1, video2, etc. 
    """
    folder_num = (frame_index // max_frames_per_folder) + 1
    folder_name = f"video{folder_num}"
    folder_path = os.path.join(output_root, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    return folder_path

# 3) Main decoding function

def decode_videos(video_files, output_root, stride=5, max_frames_per_folder=3000):
    # Ensure the output root directory exists
    os.makedirs(output_root, exist_ok=True)
    
    # Continue numbering from the last saved frame (if any)
    last_frame = get_last_frame_index(output_root)
    current_frame_index = last_frame  

    print(f"Starting at frame: {current_frame_index + 1:07d}.jpg")
    video_frame_mapping = {}  

    
    for video_file in video_files:
        cap = cv2.VideoCapture(video_file)
        if not cap.isOpened():
            print(f"Error opening video file: {video_file}")
            continue

        video_start_frame = current_frame_index + 1  # Starting frame number for this video
        frame_count = 0  # Counter for frames read from the current video

        while True:
            ret, frame = cap.read()
            if not ret:
                break  # End of the video

            # Save the frame based on the stride condition
            if frame_count % stride == 0:
                current_frame_index += 1
                folder_path = get_output_folder(current_frame_index - 1, output_root, max_frames_per_folder)
                filename = os.path.join(folder_path, f"{current_frame_index:07d}.jpg")
                cv2.imwrite(filename, frame)
            frame_count += 1

        video_end_frame = current_frame_index  # Ending frame number for this video
        video_frame_mapping[video_file] = (video_start_frame, video_end_frame)
        print(f"Video {video_file} processed: starts at {video_start_frame:07d}.jpg and ends at {video_end_frame:07d}.jpg")
        cap.release()

    return video_frame_mapping


# 4) Run the decoder on annotated videos

if not annotated_video_files:
    print("No annotated video files specified.")
else:
    mapping = decode_videos(annotated_video_files, output_root, stride, max_frames_per_folder)
    print("\nFrame range mapping per video:")
    for video, (start, end) in mapping.items():
        print(f"{video}: {start:07d}.jpg to {end:07d}.jpg")